# "Manually" working with streetscapes databases

The streetscapes CLI is an easy way to download and segment street view images.
These can all be viewed in the explorer as well.

However, for some more specific use-cases it can be necessary to work with the project database directly.
For this we have created some utilities to make it easier.

Let's start by importing the required functions from the streetscapes package:


In [ ]:
from streetscapes.utils.db_access import get_image, open_project, get_segmentations

One of these `open_project` will open the project's duckDB database using [Ibis](https://ibis-project.org/).
The database contains the tables below:

```mermaid
%% generated with https://gist.github.com/michael-simons/bd89eaae2bc8ecdcc911c1b08268894b
%% truncate large tables with …
erDiagram
    images {uuid uuid varchar source varchar shard varchar notes varchar[] tags integer rating}
    mapillary {uuid image double altitude double atomic_scale varchar camera_type ubigint captured_at … …}
    runs {varchar run timestamp timestamp varchar model json metadata}
    segmentations {varchar run boolean curated uuid image varchar[] labels integer rating geometry polygons}
```

In [ ]:
db = open_project()

We can list the different tables in the database with the following function:

In [ ]:
db.list_tables()

And inspect the table structure as follows:

In [ ]:
db.table("images")

To find the specifc data of an image, we can use the following statement to add a "predicate" for the rows where the UUID equals a certain value.
This can then be used to filter the database:

In [ ]:
predicate = db.table("images").uuid == "ccd2f50f-9cb7-4fb2-0fc0-22f0d4013a48"

images_filtered = db.table("images").filter(predicate)

The filtered table can be converted to a pandas DataFrame. It shows that there is only one result;

In [ ]:
images_filtered.to_pandas()

The same technique can be used to filter the segmentations table. The result retrieves the available segmentations for the same image:

In [ ]:
db.table("segmentations").filter(
    db.table("segmentations").image == "ccd2f50f-9cb7-4fb2-0fc0-22f0d4013a48"
).to_pandas()

In addition to the raw tables, we have already added some functions to help with the most common queries, such as retrieving an image or getting all of an image's segmentations:

In [ ]:
image = get_image("ccd2f50f-9cb7-4fb2-0fc0-22f0d4013a48")
image

In [ ]:
from rich.pretty import pprint
segmentations = get_segmentations("ccd2f50f-9cb7-4fb2-0fc0-22f0d4013a48")

pprint(segmentations,max_length=3)